[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yotamc19/Image-processing-project/blob/main/notebooks/demo_video.ipynb)


# Demo: how the model reads a handwritten word

A single word image goes in, and every stage of the network's reasoning is
shown on screen before the final text comes out. Built for the recorded demo.

| Stage | What you see |
|---|---|
| 1. Preprocessing | photo → binarized → denoised → resized to 32×128 |
| 2. STN | the model's own straightened view of the word |
| 3. Time slices | width 128 becomes 32 time steps (~4 px each) |
| 4. Per-step characters | what each step predicts, and how sure it is |
| 5. CTC collapse | raw frame string → merge repeats → drop blanks → word |

**No training here.** This notebook only loads an existing checkpoint, so it
runs in seconds and can be re-run between takes. The checkpoint comes from a
full run of `milestone3_final.ipynb` (see the *Checkpoint* section below).

Filming notes are in the last cell.


## 1. Setup


In [ ]:
import os, sys
if not os.path.exists('Image-processing-project'):
    !git clone https://github.com/yotamc19/Image-processing-project.git
%cd Image-processing-project
!pip install -q -r requirements.txt
sys.path.insert(0, '.')


## 2. Checkpoint

The weights are not in the repo (`*.pt` is gitignored). Get them once by running
`notebooks/milestone3_final.ipynb` end to end, then download
`checkpoints_m3/best_model.pt` (~33 MB) and keep it — in Drive is easiest.

The cell below uses the file if it is already there, otherwise it prompts for an
upload. To pull it from Drive instead, mount Drive and set `CHECKPOINT` to the
path inside `/content/drive/MyDrive/...`.


In [ ]:
CHECKPOINT = 'checkpoints_m3/best_model.pt'

if not os.path.exists(CHECKPOINT):
    from google.colab import files
    print('Upload the M3 checkpoint (best_model.pt):')
    uploaded = files.upload()
    os.makedirs(os.path.dirname(CHECKPOINT), exist_ok=True)
    os.replace(next(iter(uploaded)), CHECKPOINT)

print('checkpoint:', CHECKPOINT, f'({os.path.getsize(CHECKPOINT) / 1e6:.1f} MB)')


In [ ]:
from src.inference import load_model
from src.visualize import explain, run_with_intermediates

model, encoder, device = load_model(CHECKPOINT, use_stn=True)
print(f'loaded STN-CRNN on {device} — {len(encoder)} classes, '
      f'{sum(p.numel() for p in model.parameters()):,} parameters')


## 3. Pick an image

Either upload a photo of a single handwritten word, or fall back to a sample from
the held-out test split.

For a phone photo: **crop tightly around one word**, dark ink on light paper, as
straight-on as you can. The model is lowercase-only and reads one word at a time.


In [ ]:
USE_TEST_SAMPLE = True  # False -> upload your own photo

if USE_TEST_SAMPLE:
    from src.dataset import load_iam_splits
    _, _, test_hf = load_iam_splits()
    sample = test_hf[7]
    image, ground_truth = sample['image'], sample['text']
else:
    from PIL import Image
    from google.colab import files
    uploaded = files.upload()
    image, ground_truth = Image.open(next(iter(uploaded))), None

print('ground truth:', ground_truth if ground_truth else '(unknown — your own photo)')
image


## 4. The five stages

One call plots all of them in order. For filming, the cells after this one run the
same stages one at a time so you can talk over each figure.


In [ ]:
result = explain(model, image, encoder, device)
print('prediction:', repr(result['text']))
if ground_truth:
    print('ground truth:', repr(ground_truth.lower()))


### Stage by stage (one figure per cell — use these while recording)


In [ ]:
from src.visualize import (plot_preprocessing, plot_stn, plot_time_slices,
                           plot_frame_predictions, plot_ctc_collapse)

result = run_with_intermediates(model, image, encoder, device)


In [ ]:
plot_preprocessing(result)


In [ ]:
plot_stn(result)


In [ ]:
plot_time_slices(result)


In [ ]:
plot_frame_predictions(result)


In [ ]:
plot_ctc_collapse(result)


## 5. A few words in a row

Quick montage material: predictions on several test images at once, so the demo
ends on more than a single example.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from src.inference import predict
from src.dataset import load_iam_splits

_, _, test_hf = load_iam_splits()
idxs = np.random.choice(len(test_hf), 6, replace=False)
fig, axes = plt.subplots(6, 1, figsize=(12, 12))
for ax, i in zip(axes, idxs):
    s = test_hf[int(i)]
    pred = predict(model, s['image'], encoder, device)
    ax.imshow(s['image'], cmap='gray')
    mark = 'correct' if pred == s['text'].lower() else 'wrong'
    ax.set_title(f"predicted: {pred!r}   |   true: {s['text'].lower()!r}   ({mark})", fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.show()


## Filming notes

1. **Before recording** — run sections 1–3 once so the checkpoint and dataset are
   cached. Then restart nothing; just re-run the stage cells per take.
2. **Zoom the browser to ~150%** so the per-step characters are legible in the
   recorded video. Collapse the Colab file browser panel.
3. **Three images make the best story**: a clean word (it just works), a slanted or
   messy one (the STN figure visibly straightens it), and one the model gets wrong
   (explain *where* in the per-step figure the confusion happens).
4. **Record with narration live** (⌘⇧5 on macOS, pick the microphone) — matching
   voice to figures afterwards is much harder.
5. **The line worth saying out loud**: the network never sees letters, only 32
   vertical strips; the word only appears at the CTC collapse, when repeated
   predictions merge and blanks are dropped.
